In [3]:
gameLog = [('PHI','2009-02-07'),
          ('WAS','2017-02-13'),
          ('OKC','2017-04-04'),
          ('DEN','2017-04-09'),
          ('GSW','2018-02-24'),
          ('SAC','2018-11-19'),
          ('GSW','2018-11-21'),
          ('GSW','2019-11-25'),
          ('SAC','2019-12-11'),
          ('BKN','2020-01-07'),
          ('HOU','2020-01-15'),
          ('HOU','2020-01-20'),
          ('PHI','2021-04-26'),
          ('WAS','2021-05-29'),
          ('LAC','2021-11-01'),
          ('LAL','2021-11-02'),
          ('LAL','2021-11-04'),
          ('OKC','2021-11-12'),
          ('LAL','2023-02-07'),
          ('PHX','2023-02-24'),
          ('LAC','2023-03-18'),
          ('LAC','2023-03-21'),
          ('LAL','2024-01-15'),
          ('LAC','2024-01-16'),
          ('LAC','2024-03-01'),
          ('LAL','2024-03-04'),
          ('DEN','2024-10-24'),
          ('DEN','2024-10-26'),
          ('LAC','2024-11-02'),
          ('LAL','2024-11-29'),
          ('LAC','2024-12-01'),
          ('PHX','2025-01-12'),
          ('LAC','2025-03-23'),
          ('LAC','2025-04-26'),
          ('DEN','2025-05-11'),
          ('OKC','2025-06-08'),
          ('OKC','2025-06-22'),
          ('LAC','2025-11-04'),
          ('SAC','2025-11-07'),
          ('ORL','2025-12-13'),
          ('OKC','2025-12-13'),
          ('PHX','2026-01-02'),
          ('PHX','2026-01-04'),
          ('LAC','2026-03-14'),
          ('LAL','2026-05-09'),
          ('SAS','2026-05-24'),
          ('LAL','2026-11-02')]

newGames = [('LAC','2026-11-13')]
print(len(gameLog),'+',len(newGames)) 

47 + 1


In [160]:
from bs4 import BeautifulSoup, Comment
import pandas as pd
import re
import requests

def boxScoreScrape(team,date):
    boxscore_data = pd.DataFrame()
    player_data = pd.DataFrame()
    gameDate = date.replace('-','')
    homeTeam=team
    url = 'https://www.basketball-reference.com/boxscores/%s0%s.html' % (gameDate,homeTeam)
    print(url)
    headers = {
      "User-Agent": (
          "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
          " like Gecko) Chrome/120.0.0.0 Safari/537.36"
      )
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to fetch page. Status code: {response.status_code}")
        return None
    
    html_content = response.text
    uncommented_html = html_content.replace("<!--", "").replace("-->", "")
    soup = BeautifulSoup(uncommented_html, "html.parser")

    team_elements = soup.select('div.scorebox strong a')
    team_names = [team.text.strip() for team in team_elements]
    awayTeamName = team_names[0]
    homeTeamName = team_names[1]
    scorebox = soup.find(attrs = {'class' : 'scorebox'})
    scores = scorebox.find_all(attrs = {'class' : 'score'})
    awayTeamScore = int(scores[0].getText())
    homeTeamScore = int(scores[1].getText())
    if awayTeamScore > homeTeamScore:
        awayTeamResult = 'W'
        homeTeamResult = 'L'
    else:
        homeTeamResult = 'W'
        awayTeamResult = 'L'    
    
    try: 
        teamLinks=scorebox.select("a[href*=teams]")
    except:
        print(hteam,gameDate)
        return (None,None)
        
    awayTeamAbv = (teamLinks[0]['href'].split('/')[2])

    #################
    # game info #
    #################
        
    gameMetadata = soup.find(attrs = {'class' : 'scorebox_meta'})
    gameMetadataList = gameMetadata.find_all('div')
    
    if 'Cup' in gameMetadataList[0].text:
        gameNote = 'NBA Cup'
        gameDate = gameMetadataList[1].text.split('M, ')[1]
        gameArena = gameMetadataList[2].text.split(',')[0]
        gameCity = gameMetadataList[2].text.split(',')[1]
        
    else:
        gameNote = 'Regular Season'
        gameDate = gameMetadataList[0].text.split('M, ')[1]
        gameArena = gameMetadataList[1].text.split(',')[0]
        gameCity = gameMetadataList[1].text.split(',')[1]

    gameDate = pd.to_datetime(gameDate).strftime("%Y-%m-%d")

    gameInfoList_home = [team, homeTeamName, gameDate, f"{team} vs. {awayTeamAbv}", homeTeamResult,  gameArena, gameCity, gameNote]
    gameInfoList_away = [awayTeamAbv, awayTeamName, gameDate, f"{awayTeamAbv} @ {team}", awayTeamResult, gameArena, gameCity, gameNote]
    
    #################
    # player info #
    #################
    
    # Find all basic box score tables using regex matching IDs
    tables = soup.find_all("table", {"id": re.compile(r"box-.*-game-basic")})

    boxscore = pd.DataFrame()
    player_stats = pd.DataFrame()
    
    for table in tables:
        table_id = table.get("id")
        # Extract the team abbreviation from the table ID (e.g., 'box-LAC-game-basic' -> 'LAC')
        team_abbr = table_id.split("-")[1]
    
        # Extract headers from <thead>
        thead = table.find("thead")
        columns = []
        if thead:
          header_rows = thead.find_all("tr")
          if header_rows:
            columns = [th.get_text(strip=True) for th in header_rows[-1].find_all(["th", "td"])]
        
        # Extract rows from <tbody>
        tbody = table.find("tbody")
        rows_data = []
        if tbody:
          for tr in tbody.find_all("tr"):
            cells = tr.find_all(["th", "td"])
            row_values = [cell.get_text(strip=True) for cell in cells]
            if row_values and any(row_values):
              rows_data.append(row_values)

        # Extract rows from <tfoot>
        tfoot = table.find("tfoot")
        foot_data = []
        if tfoot:
          for tr in tfoot.find_all("tr"):
            cells = tr.find_all(["th", "td"])
            foot_values = [cell.get_text(strip=True) for cell in cells]
            if foot_values and any(foot_values):
              if team_abbr == team:
                foot_values.extend(gameInfoList_home)
              else:
                foot_values.extend(gameInfoList_away)
              foot_data.append(foot_values)

        if foot_data:
          totals = pd.DataFrame(foot_data)
          bs_columns = columns + (["ABBREV","TEAM","DATE","MATCHUP","W/L","ARENA","CITY","NOTE"])
          totals.columns = bs_columns
          boxscore = pd.concat([boxscore,totals], ignore_index=True)

        if rows_data:
          df = pd.DataFrame(rows_data)

          if columns and len(columns) == df.shape[1]:
            df.columns = columns
              
          df = df.dropna(how="all")
          if "Starters" in df.columns:
            df = df[df["Starters"] != "Reserves"]
            df = df[df["Starters"] != "Team Totals"]
          df = df[df["MP"] != "Did Not Play"]

          df["TEAM"] = team_abbr
          df["DATE"] = gameDate
          if team_abbr == team:
              df["OPP"] = awayTeamAbv
          else:
              df["OPP"] = team
          df["G"] = 1
          df["REB"] = df["ORB"].astype(int) + df["DRB"].astype(int)
          # 1. Split the 'MP' column by the colon into temporary columns (handling non-strings safely)
          split_cols = df["MP"].astype(str).str.split(":", expand=True)
          mins = pd.to_numeric(split_cols[0], errors="coerce")
          secs = (pd.to_numeric(split_cols[1], errors="coerce").fillna(0) if split_cols.shape[1] > 1 else pd.Series(0, index=df.index))
          df["MP"] = mins + (secs >= 30).astype(int)
            
          player_stats = pd.concat([player_stats,df], ignore_index=True)
            
    player_stats = player_stats[["Starters","TEAM","DATE","OPP","MP","PTS","REB","AST","STL","BLK","FG","FGA","FG%","3P","3PA","FT","FTA","DRB","ORB","TOV","PF","G"]]
    player_stats = player_stats.rename(columns={"Starters": "NAME", "MP": "MINS", "FG": "FGM", "FT": "FTM","3P": "3PM",  "DRB": "DREB", "ORB": "OREB", "TOV": "TO"})
    player_stats = player_stats.astype({"MINS":int,"PTS":int,"REB":int,"AST":int,"STL":int,"BLK":int,"FGM":int,"FGA":int,"3PM":int,"3PA":int,"FTM":int,"FTA":int,"DREB":int,"OREB":int,"TO":int,"PF":int,"FG%":float})
    
    boxscore.drop(columns=['Starters', 'MP', 'GmSc','+/-'])
    boxscore = boxscore[["ABBREV","TEAM","DATE","MATCHUP","W/L","PTS","FG","FGA","FG%","3P","3PA","3P%","FT","FTA","FT%","ORB","DRB","TRB","AST","STL","BLK","TOV","PF","ARENA","CITY","NOTE"]]
    boxscore = boxscore.rename(columns={"FG": "FGM", "FT": "FTM", "DRB": "DREB", "ORB": "OREB", "TRB":"REB","3P":"3PM","TOV":"TO"})
    boxscore = boxscore.astype({"PTS":int,"REB":int,"AST":int,"STL":int,"BLK":int,"FGM":int,"FGA":int,"3PM":int,"3PA":int,"FTM":int,"FTA":int,"DREB":int,"OREB":int,"TO":int,"PF":int,"FG%":float,"FT%":float,"3P%":float})
    
    return boxscore, player_stats

In [161]:
for game in newGames:
    boxscore, player_stats = boxScoreScrape(game[0],game[1])
pd.set_option("display.max_columns", 30)
display(boxscore)
display(player_stats)

https://www.basketball-reference.com/boxscores/202611130LAC.html


,ABBREV,TEAM,DATE,MATCHUP,W/L,PTS,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,OREB,DREB,REB,AST,STL,BLK,TO,PF,ARENA,CITY,NOTE
0,OKC,Oklahoma City Thunder,2026-11-13,OKC @ LAC,L,112,44,94,0.468,11,35,0.314,13,19,0.684,11,34,45,27,8,4,13,17,Intuit Dome,Inglewood,NBA Cup
1,LAC,Los Angeles Clippers,2026-11-13,LAC vs. OKC,W,113,40,88,0.455,16,41,0.390,17,21,0.810,10,30,40,28,7,7,11,20,Intuit Dome,Inglewood,NBA Cup


,NAME,TEAM,DATE,OPP,MINS,PTS,REB,AST,STL,BLK,FGM,FGA,FG%,3PM,3PA,FTM,FTA,DREB,OREB,TO,PF,G
0,Shai Gilgeous-Alexander,OKC,2026-11-13,LAC,39,40,4,5,1,0,15,26,0.577,2,4,8,11,3,1,3,1,1
1,Kenrich Williams,OKC,2026-11-13,LAC,35,8,8,4,3,0,4,11,0.364,0,4,0,0,6,2,0,0,1
2,Jared McCain,OKC,2026-11-13,LAC,31,23,4,2,0,0,8,17,0.471,7,11,0,0,4,0,2,5,1
3,Isaiah Hartenstein,OKC,2026-11-13,LAC,30,21,16,6,1,0,9,10,0.900,0,0,3,4,12,4,2,4,1
4,Aday Mara,OKC,2026-11-13,LAC,21,10,3,0,0,3,5,6,0.833,0,0,0,0,3,0,2,2,1
5,Alex Caruso,OKC,2026-11-13,LAC,27,5,1,4,1,1,1,9,0.111,1,6,2,2,1,0,1,0,1
6,Josh Dix,OKC,2026-11-13,LAC,17,0,4,3,0,0,0,6,0.000,0,4,0,0,1,3,0,1,1
7,Nikola TopiÄ,OKC,2026-11-13,LAC,16,0,4,3,2,0,0,4,0.000,0,2,0,0,3,1,2,1,1
8,Bennett Stirtz,OKC,2026-11-13,LAC,12,5,1,0,0,0,2,4,0.500,1,3,0,0,1,0,0,1,1
9,Nique Clifford,OKC,2026-11-13,LAC,12,0,0,0,0,0,0,1,0.000,0,1,0,2,0,0,0,2,1


In [162]:
#playerStats.to_csv('/storage/emulated/0/proj/newstats.csv')
#boxScores.to_csv('/storage/emulated/0/proj/newbox.csv')
#playerStats.to_csv('newstats.csv')
#boxScores.to_csv('newbox.csv')

In [152]:
all_boxes = pd.read_csv('nba_boxscores.csv',index_col=[0])
all_stats = pd.read_csv('nba_statlines.csv',index_col=[0])

In [163]:
all_boxes

,ABBREV,TEAM,DATE,MATCHUP,W/L,PTS,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,OREB,DREB,REB,AST,STL,BLK,TO,PF,ARENA,CITY,NOTE
0,PHI,Philadelphia 76ers,2009-02-07,PHI vs. MIA,W,94,30,69,0.435,4,14,0.286,30,36,0.833,9,31,40,21,7,6,11,18,Wells Fargo Center,Philadelphia,Regular Season
1,MIA,Miami Heat,2009-02-07,MIA @ PHI,L,84,34,77,0.442,3,19,0.158,13,20,0.650,9,26,35,23,4,1,15,25,Wells Fargo Center,Philadelphia,Regular Season
2,WAS,Washington Wizards,2017-02-13,WAS vs. OKC,W,120,42,73,0.575,13,20,0.650,23,28,0.821,6,34,40,27,4,9,22,24,Capital One Arena,"Washington, D.C.",Regular Season
3,OKC,Oklahoma City Thunder,2017-02-13,OKC @ WAS,L,98,34,96,0.354,7,34,0.206,23,30,0.767,20,27,47,16,10,2,15,22,Capital One Arena,"Washington, D.C.",Regular Season
4,OKC,Oklahoma City Thunder,2017-04-04,OKC vs. MIL,W,110,40,84,0.476,9,28,0.321,21,25,0.840,11,42,53,29,9,7,13,20,Chesapeake Energy Arena,Oklahoma City,Regular Season
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,OKC,Oklahoma City Thunder,2026-05-09,OKC @ LAL,W,131,53,94,0.564,17,38,0.447,8,10,0.800,12,31,43,30,12,3,9,22,Crypto.com Arena,Los Angeles,Playoffs R2 G3
90,SAS,San Antonio Spurs,2026-05-24,SAS vs. OKC,W,103,35,90,0.389,9,33,0.273,24,32,0.750,14,38,52,25,11,10,13,20,Frost Bank Center,San Antonio,WCF G4
91,OKC,Oklahoma City Thunder,2026-05-24,OKC @ SAS,L,82,30,91,0.330,6,33,0.182,16,18,0.889,11,36,47,22,9,5,20,25,Frost Bank Center,San Antonio,WCF G4
92,LAL,Los Angeles Lakers,2026-11-02,LAL vs. MIA,W,138,51,89,0.573,19,43,0.442,17,24,0.708,10,31,41,39,8,4,16,28,Crypto.com Arena,Los Angeles,Regular Season


In [169]:
all_stats

,NAME,TEAM,DATE,OPP,MINS,PTS,REB,AST,STL,BLK,FGM,FGA,FG%,3PM,3PA,FTM,FTA,DREB,OREB,TO,PF,G
0,Mario Chalmers,MIA,2009-02-07,PHI,42,18,5,7,2,0,7,15,0.470,2,6,2,5,3,2,2,3,1
1,Dwyane Wade,MIA,2009-02-07,PHI,41,21,4,7,0,0,8,19,0.420,0,4,5,8,3,1,2,3,1
2,Shawn Marion,MIA,2009-02-07,PHI,40,11,6,2,0,1,3,8,0.380,0,0,5,5,5,1,3,2,1
3,Udonis Haslem,MIA,2009-02-07,PHI,40,8,9,2,1,0,4,11,0.360,0,0,0,0,7,2,0,4,1
4,Jamaal Magloire,MIA,2009-02-07,PHI,15,5,3,2,0,0,2,2,1.000,0,0,1,2,3,0,2,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1011,Dru Smith,MIA,2026-11-02,LAL,11,3,3,2,2,0,0,2,0.000,0,2,3,4,2,1,0,0,1
1012,Ryan Conwell,MIA,2026-11-02,LAL,8,5,1,1,1,0,2,4,0.500,1,3,0,0,0,1,0,2,1
1013,Nikola Jovic,MIA,2026-11-02,LAL,7,2,1,0,0,0,1,3,0.333,0,2,0,0,1,0,1,1,1
1014,Myron Gardner,MIA,2026-11-02,LAL,1,0,0,0,0,0,0,0,0.000,0,0,0,0,0,0,0,0,1


In [172]:
updated_box = pd.concat([all_boxes,boxscore], ignore_index=True)
updated_stats = pd.concat([all_stats,player_stats], ignore_index=True)

In [173]:
updated_box

,ABBREV,TEAM,DATE,MATCHUP,W/L,PTS,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,OREB,DREB,REB,AST,STL,BLK,TO,PF,ARENA,CITY,NOTE
0,PHI,Philadelphia 76ers,2009-02-07,PHI vs. MIA,W,94,30,69,0.435,4,14,0.286,30,36,0.833,9,31,40,21,7,6,11,18,Wells Fargo Center,Philadelphia,Regular Season
1,MIA,Miami Heat,2009-02-07,MIA @ PHI,L,84,34,77,0.442,3,19,0.158,13,20,0.650,9,26,35,23,4,1,15,25,Wells Fargo Center,Philadelphia,Regular Season
2,WAS,Washington Wizards,2017-02-13,WAS vs. OKC,W,120,42,73,0.575,13,20,0.650,23,28,0.821,6,34,40,27,4,9,22,24,Capital One Arena,"Washington, D.C.",Regular Season
3,OKC,Oklahoma City Thunder,2017-02-13,OKC @ WAS,L,98,34,96,0.354,7,34,0.206,23,30,0.767,20,27,47,16,10,2,15,22,Capital One Arena,"Washington, D.C.",Regular Season
4,OKC,Oklahoma City Thunder,2017-04-04,OKC vs. MIL,W,110,40,84,0.476,9,28,0.321,21,25,0.840,11,42,53,29,9,7,13,20,Chesapeake Energy Arena,Oklahoma City,Regular Season
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,OKC,Oklahoma City Thunder,2026-05-24,OKC @ SAS,L,82,30,91,0.330,6,33,0.182,16,18,0.889,11,36,47,22,9,5,20,25,Frost Bank Center,San Antonio,WCF G4
92,LAL,Los Angeles Lakers,2026-11-02,LAL vs. MIA,W,138,51,89,0.573,19,43,0.442,17,24,0.708,10,31,41,39,8,4,16,28,Crypto.com Arena,Los Angeles,Regular Season
93,MIA,Miami Heat,2026-11-02,MIA @ LAL,L,117,38,75,0.507,16,41,0.390,25,35,0.714,7,27,34,28,7,5,19,23,Crypto.com Arena,Los Angeles,Regular Season
94,OKC,Oklahoma City Thunder,2026-11-13,OKC @ LAC,L,112,44,94,0.468,11,35,0.314,13,19,0.684,11,34,45,27,8,4,13,17,Intuit Dome,Inglewood,NBA Cup


In [174]:
updated_stats

,NAME,TEAM,DATE,OPP,MINS,PTS,REB,AST,STL,BLK,FGM,FGA,FG%,3PM,3PA,FTM,FTA,DREB,OREB,TO,PF,G
0,Mario Chalmers,MIA,2009-02-07,PHI,42,18,5,7,2,0,7,15,0.470,2,6,2,5,3,2,2,3,1
1,Dwyane Wade,MIA,2009-02-07,PHI,41,21,4,7,0,0,8,19,0.420,0,4,5,8,3,1,2,3,1
2,Shawn Marion,MIA,2009-02-07,PHI,40,11,6,2,0,1,3,8,0.380,0,0,5,5,5,1,3,2,1
3,Udonis Haslem,MIA,2009-02-07,PHI,40,8,9,2,1,0,4,11,0.360,0,0,0,0,7,2,0,4,1
4,Jamaal Magloire,MIA,2009-02-07,PHI,15,5,3,2,0,0,2,2,1.000,0,0,1,2,3,0,2,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1031,Max Strus,LAC,2026-11-13,OKC,23,3,2,3,4,0,0,4,0.000,0,2,3,3,2,0,1,3,1
1032,Gradey Dick,LAC,2026-11-13,OKC,19,13,3,0,0,0,5,7,0.714,2,2,1,3,2,1,0,1,1
1033,Keaton Wagler,LAC,2026-11-13,OKC,18,14,2,1,0,0,4,7,0.571,2,5,4,4,2,0,2,2,1
1034,Isaiah Jackson,LAC,2026-11-13,OKC,17,5,3,3,0,1,2,6,0.333,0,2,1,2,1,2,1,2,1


In [175]:
updated_box.to_csv('nba_boxscores.csv')
updated_stats.to_csv('nba_statlines.csv')

### outdated

In [1]:
import pandas as pd

from nba_api.stats.static import teams
from nba_api.stats.library.parameters import Season
from nba_api.stats.library.parameters import SeasonType
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.live.nba.endpoints import boxscore
from nba_api.stats.endpoints import boxscoretraditionalv3

In [2]:
# Data Sets for
# box = boxscore.BoxScore(game_id)
# ---------------------------------
#box.game.get_dict()                    #equal to box.get_dict()['game']
#box.arena.get_dict()                  #equal to box.get_dict()['game']['arena']
#box.away_team.get_dict()              #equal to box.get_dict()['game']['awayTeam']
#box.away_team_player_stats.get_dict() #equal to box.get_dict()['game']['awayTeam']['players']
#box.away_team_stats.get_dict()        #equal to box.get_dict()['game']['homeTeam'] w/o ['players']
#box.home_team.get_dict()              #equal to box.get_dict()['game']['homeTeam']
#box.home_team_player_stats.get_dict() #equal to box.get_dict()['game']['homeTeam']['players']
#box.home_team_stats.get_dict()        #equal to box.get_dict()['game']['homeTeam'] w/o ['players']
#box.game_details.get_dict()           #equal to box.get_dict()['game'] scrubbed of all other dictionaries
#box.officials.get_dict()              #equal to box.get_dict()['game']['officials']

In [3]:
NBA_STATS_HEADERS: dict[str, str] = {
"Accept": "application/json, text/plain, /",
"Accept-Encoding": "gzip, deflate, br",
"Accept-Language": "en-US,en;q=0.9",
"Cache-Control": "no-cache",
"Connection": "keep-alive",
"Host": "stats.nba.com",
"Origin": "https://www.nba.com",
"Pragma": "no-cache",
"Referer": "https://www.nba.com/",
"Sec-Ch-Ua": '"Google Chrome";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
"Sec-Ch-Ua-Mobile": "?0",
"Sec-Ch-Ua-Platform": '"Windows"',
"Sec-Fetch-Dest": "empty",
"Sec-Fetch-Mode": "cors",
"Sec-Fetch-Site": "same-site",
"User-Agent": (
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
"AppleWebKit/537.36 (KHTML, like Gecko) "
"Chrome/131.0.0.0 Safari/537.36"
),
}
try:
    from nba_api.stats.library import http as stats_http
    from nba_api.library import http as base_http

    stats_http.STATS_HEADERS = NBA_STATS_HEADERS          # global constant
    stats_http.NBAStatsHTTP.headers = NBA_STATS_HEADERS    # class default
    # Reset any existing session so stale connections are dropped
    stats_http.NBAStatsHTTP._session = None
    base_http.NBAHTTP._session = None
except Exception:
    pass  # nba_api not installed — nothing to patch

In [4]:
#result = leaguegamefinder.LeagueGameFinder(season_nullable='2025-26',league_id_nullable='00')
result = leaguegamefinder.LeagueGameFinder()
all_games = result.get_data_frames()[0]

In [5]:
def boxpuller(homeTeam, date):

    stupid=date.split('-')
    stupidDate = stupid[1]+'/'+stupid[2]+'/'+stupid[0]
    
    nba_teams = teams.get_teams()

    teamA = [team for team in nba_teams if team['abbreviation'] == homeTeam][0]
    teamA_id = teamA['id']

    gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=teamA_id,
                                                   date_from_nullable=stupidDate,
                                                   date_to_nullable=stupidDate) 
    
    game = gamefinder.get_data_frames()[0].sort_values('GAME_DATE').iloc[-1]
    
    game_id = game.GAME_ID
    matchup = game.MATCHUP
    print(f'Getting boxscore for {matchup} on {date}')

    # Find the game_id we want
    full_game = all_games[all_games.GAME_ID == game_id]
    full_game = full_game.drop(columns=['SEASON_ID','TEAM_ID','GAME_ID','MIN','PLUS_MINUS'])
    
    awayTeam = matchup.split()[2]
    all_player_stats = []
    try:
        #print(game_id)
        custom_headers = {
            'Host': 'stats.nba.com',
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Accept-Language': 'en-US,en;q=0.5',
            'Referer': 'https://nba.com',
            'Connection': 'keep-alive'}
        box = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id,headers=custom_headers)
        print(box)
        #box = boxscore.BoxScore(game_id,headers=custom_headers)
    except:
        print('boxscore issue')
        return
    try:
        skibby = box.game.get_dict()
        #print(skibby["attendance"])
        location = box.arena.get_dict()
        arena = location['arenaName']
        city = location['arenaCity']
        full_game['ARENA'] = arena
        full_game['CITY'] = city
        
        ht_stats = box.home_team_player_stats.get_dict()
        for player in ht_stats:
            if player['status'] == 'ACTIVE':
                player_stats = player['statistics']
                player_stats.update({"NAME": player['name'],"TEAM":homeTeam, "OPP":awayTeam, "DATE":date})
                all_player_stats.append(player_stats)
                
        at_stats = box.away_team_player_stats.get_dict()
        for player in at_stats:
            if player['status'] == 'ACTIVE':
                player_stats = player['statistics']
                player_stats.update({"NAME": player['name'],"TEAM":awayTeam, "OPP":homeTeam, "DATE":date})
                all_player_stats.append(player_stats)
    except:
        print('cant get player data')    
        box = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id,headers=custom_headers)
        all_player_stats = []
        pstats = box.data_sets[0].get_dict()
        pstats = pstats.values()
        ps = pd.DataFrame.from_dict(pstats)
        #print(ps)
        bs = pd.DataFrame(columns=pstats.keys)
        for player in pstats:
            print(player)
            if player['status'] == 'ACTIVE':
               player_stats = player['statistics']
               player_stats.update({"NAME": player['name'],"TEAM":homeTeam, "OPP":awayTeam, "DATE":date})
               all_player_stats.append(player_stats)  

    return (full_game, all_player_stats)

In [6]:
def get_all_stats(games):
    
    
    gameLogPlayerStats = []
    boxes=pd.DataFrame()

    for team,date in games:
        print(team,date)
        boxInfo, gamePlayerStats = boxpuller(team,date)
        boxes = pd.concat([boxes,boxInfo], ignore_index=True)
        for playerStats in gamePlayerStats:
            gameLogPlayerStats.append(playerStats)

    try:


        boxes = boxes.rename(columns={"TEAM_ABBREVIATION":"ABBREV","TEAM_NAME":"TEAM","GAME_DATE":"DATE",'WL':'W/L', 'FG_PCT':'FG%',
                                      'FG3M':'3PM', 'FG3A':'3PA', 'FG3_PCT':'3P%','FT_PCT':'FT%'})

        
        all_stats = pd.DataFrame(gameLogPlayerStats)
        all_stats = all_stats.drop(columns=['blocksReceived', 'foulsOffensive','foulsDrawn','minus',
                            'plus','plusMinusPoints','pointsFastBreak','pointsInThePaint','pointsSecondChance',
                                 'twoPointersAttempted','twoPointersMade','twoPointersPercentage',
                             'threePointersPercentage','freeThrowsPercentage','foulsTechnical'])
    
        all_stats = all_stats[['NAME','TEAM','DATE','OPP', 'minutesCalculated', 'points','reboundsTotal','assists','steals','blocks',
         'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
         'threePointersMade', 'threePointersAttempted',
         'freeThrowsMade', 'freeThrowsAttempted',
         'reboundsDefensive','reboundsOffensive',
         'turnovers',
         'foulsPersonal']]
    
        all_stats = all_stats.rename(columns={"minutesCalculated": "MINS", "points": "PTS", "reboundsTotal": "REB",
                               "assists": "AST","steals": "STL","blocks": "BLK",
                                "fieldGoalsMade": "FGM", "fieldGoalsAttempted": "FGA", "fieldGoalsPercentage": "FG%",
                               "threePointersMade": "3P", "threePointersAttempted": "3PA",
                               "freeThrowsMade": "FTM", "freeThrowsAttempted": "FTA",
                                "reboundsDefensive":"dREB", "reboundsOffensive":"oREB","turnovers":"TO",
                               "foulsPersonal":"FLS"})
    
        all_stats = all_stats.round({'FG%': 2})
        all_stats['MINS'] = all_stats['MINS'].str.replace(r'PT', '')
        all_stats['MINS'] = all_stats['MINS'].str.replace(r'M', '')
        all_stats = all_stats[all_stats['MINS'] != '00']
        all_stats['G'] = 1
    except:
        print('no player data')
    
    print()
    print("Done!")
    
    return (boxes, all_stats)

In [8]:
#boxScores, playerStats = get_all_stats(gameLog)

In [10]:
boxScores, playerStats = get_all_stats(newGames)